# 🧹 Milestone 2: Data Cleaning & Aggregation

**Project:** Network Intrusion Detection System (CICIDS-2017)  
**Team:** Bassem Gamal Mohamed – Dina Fawzy – Mohamed Farag – Pola Mokhtar – Ahmed Mamdouh – Mustafa Ibrahim  
**Team Leader:** Ahmed Mamdouh
**Github Repo:** https://github.com/AhmedELNajjar-dev/Depi-Final-Project  
**Google Drive:** https://drive.google.com/drive/folders/1wa5Y5heNduaFH-T2bIkQAkoXlhiy0QCe?usp=sharing
**Goal:** To consolidate the raw CSV files into a single, high-quality dataset by removing errors, handling duplicates, and optimizing memory usage.

### 📋 Overview of Cleaning Phases
In this milestone, we prepare the raw data through 10 distinct phases:
1.  **Initialization:** Import necessary libraries.
2.  **Data Aggregation:** Load and merge the 8 separate daily traffic files into one DataFrame.  
3.  **Metadata Removal:** Drop non-predictive identifiers like `Flow ID`.
4.  **Exploratory Data Inspection:** Examine the dataset structure, data types, and statistical summaries to identify potential issues.
5.  **Data Sanitization & Integrity:** Remove invalid entries (NaN/Infinity) and filter out logically impossible values (e.g., negative flow durations).
6.  **Duplicate Removal:** Remove **exact duplicates** only (rows identical in every column) to ensure data uniqueness while retaining repetitive attack patterns.
7.  **Timestamp Standardization:** Convert time data into a standard datetime object for temporal analysis.
8.  **Memory Optimization:** Downcast integer columns to smaller types (e.g., `int64` $\to$ `int8`) to reduce memory footprint.
9.  **Label Verification:** Ensure the target labels are consistent and correctly formatted.
10. **Final Output:** Save the clean dataset to `Cleaned_Data.parquet`.

# 1️⃣ Phase 1 & 2: Initialization and Data Aggregation
**Objective:** Combine the scattered raw data into a single workspace.
**Action:** We iterate through the 8 source CSV files (covering Monday–Friday), load them, and concatenate them into one massive DataFrame to establish a unified dataset for processing.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make pandas tables easier to read
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)    # show all rows in Series/DataFrame
pd.set_option('display.max_columns', None) # show all columns

## Loading all csv files and reading them


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Specify the path and filename for the output CSV file
path = "/content/drive/MyDrive/Depi Project/Data/Pre_Cleaning_Data.csv"

# Save the DataFrame to CSV
df_full = pd.read_csv(path, encoding="utf-8", low_memory=False)
df_full.columns = df_full.columns.str.strip()


In [ ]:
print(f"Original shape of all files combined: {df_full.shape}")

Original shape of all files combined: (3119345, 85)


# 2️⃣ Phase 3: Exploratory Data Inspection
**Objective:** To understand the structure and statistical properties of the aggregated data.
**Action:** We utilize:
* `head()`: To view sample rows and understand data format.
* `info()`: To check data types and non-null counts.
* `describe()`: To generate statistical summaries (mean, min, max) which help in detecting anomalies, such as negative values in positive-only features.

In [ ]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3119345 entries, 0 to 3119344
Data columns (total 85 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Flow ID                      object 
 1   Source IP                    object 
 2   Source Port                  float64
 3   Destination IP               object 
 4   Destination Port             float64
 5   Protocol                     float64
 6   Timestamp                    object 
 7   Flow Duration                float64
 8   Total Fwd Packets            float64
 9   Total Backward Packets       float64
 10  Total Length of Fwd Packets  float64
 11  Total Length of Bwd Packets  float64
 12  Fwd Packet Length Max        float64
 13  Fwd Packet Length Min        float64
 14  Fwd Packet Length Mean       float64
 15  Fwd Packet Length Std        float64
 16  Bwd Packet Length Max        float64
 17  Bwd Packet Length Min        float64
 18  Bwd Packet Length Mean       float64
 19  

In [ ]:
print("Descriptive statistics for numeric features:")
df_full.describe()

Descriptive statistics for numeric features:


,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.829385e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2830743.0,2.830743e+06,2830743.0,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2830743.0,2830743.0,2830743.0,2830743.0,2830743.0,2830743.0,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06
mean,4.112886e+04,8.071483e+03,9.880341e+00,1.478566e+07,9.361160e+00,1.039377e+01,5.493024e+02,1.616264e+04,2.075999e+02,1.871366e+01,5.820194e+01,6.891013e+01,8.708495e+02,4.104958e+01,3.059493e+02,3.353257e+02,inf,inf,1.298449e+06,2.919271e+06,9.182475e+06,1.623796e+05,1.448296e+07,2.610193e+06,3.266957e+06,9.042939e+06,1.021893e+06,9.893830e+06,1.805784e+06,1.485973e+06,4.684692e+06,9.672614e+05,4.644646e-02,0.0,1.112782e-04,0.0,-2.599739e+04,-2.273275e+03,6.386535e+04,6.995192e+03,1.643450e+01,9.504024e+02,1.719444e+02,2.949756e+02,4.861548e+05,3.537976e-02,4.644646e-02,2.423392e-04,2.980705e-01,3.158443e-01,9.482316e-02,1.112782e-04,2.433990e-04,6.835004e-01,1.919837e+02,5.820194e+01,3.059493e+02,-2.599739e+04,0.0,0.0,0.0,0.0,0.0,0.0,9.361160e+00,5.492919e+02,1.039377e+01,1.616230e+04,6.989837e+03,1.989433e+03,5.418218e+00,-2.741688e+03,8.155132e+04,4.113412e+04,1.531825e+05,5.829582e+04,8.316037e+06,5.038439e+05,8.695752e+06,7.920031e+06
std,2.229494e+04,1.828363e+04,5.261922e+00,3.365374e+07,7.496728e+02,9.973883e+02,9.993589e+03,2.263088e+06,7.171848e+02,6.033935e+01,1.860912e+02,2.811871e+02,1.946367e+03,6.886260e+01,6.052568e+02,8.396932e+02,NaN,NaN,4.507944e+06,8.045870e+06,2.445954e+07,2.950282e+06,3.357581e+07,9.525722e+06,9.639055e+06,2.452916e+07,8.591436e+06,2.873661e+07,8.887197e+06,6.278469e+06,1.716095e+07,8.308983e+06,2.104500e-01,0.0,1.054826e-02,0.0,2.105286e+07,1.452209e+06,2.475371e+05,3.815170e+04,2.523772e+01,2.028229e+03,3.054915e+02,6.318001e+02,1.647490e+06,1.847378e-01,2.104500e-01,1.556536e-02,4.574107e-01,4.648513e-01,2.929706e-01,1.054826e-02,1.559935e-02,6.804920e-01,3.318603e+02,1.860912e+

Identifying Redundant Features for Next Milestone:

Looking back at the `.describe()` output, we can see several features where the `std` (standard deviation) is 0. This means the `min`, `max`, and `mean` are all identical, and the feature has zero variance.

Specifically, these are columns like `Bwd PSH Flags`, `Bwd URG Flags`, `Fwd Avg Bytes/Bulk`, and others. These features provide no predictive value.

these "redundant values" will be dropped in the **Milestone 4: Data Preprocessing** phase.

In [ ]:
df_full.head()

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.5-8.254.250.126-49188-80-6,8.254.250.126,80.0,192.168.10.5,49188.0,6.0,03/07/2017 08:55:58,4.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,3000000.0,5.000000e+05,4.0,0.0,4.0,4.0,4.0,4.0,0.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,5.000000e+05,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,9.0,6.0,0.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,12.0,0.0,0.0,329.0,-1.0,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,192.168.10.5-8.254.250.126-49188-80-6,8.254.250.126,80.0,192.168.10.5,49188.0,6.0,03/07/2017 08:55:58,1.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,12000000.0,2.000000e+06,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,2.000000e+06,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,9.0,6.0,0.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,12.0,0.0,0.0,329.0,-1.0,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,192.168.10.5-8.254.250.126-49188-80-6,8.254.250.126,80.0,192.168.10.5,49188.0,6.0,03/07/2017 08:55:58,1.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,12000000.0,2.000000e+06,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,2.000000e+06,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,9.0,6.0,0.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,12.0,0.0,0.0,329.0,-1.0,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,192.168.10.5-8.254.250.126-49188-80-6,8.254.250.126,80.0,192.168.10.5,49188.0,6.0,03/07/2017 08:55:58,1.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,12000000.0,2.000000e+06,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,2.000000e+06,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,9.0,6.0,0.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,12.0,0.0,0.0,329.0,-1.0,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,192.168.10.14-8.253.185.121-49486-80-6,8.253.185.121,80.0,192.168.10.14,49486.0,6.0,03/07/2017 08:56:22,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,4000000.0,6.666667e+05,3.0,0.0,3.0,3.0,3.0,3.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,6.666667e+05,0.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,9.0,6.0,0.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,12.0,0.0,0.0,245.0,-1.0,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


# 3️⃣ Phase 5: Remove Non-Predictive Metadata
**Objective:** Remove unique identifiers that do not contribute to behavioral analysis.  
**Action:** We drop the `Flow ID` column. This is a randomly generated identifier for the connection and provides no generalizable information about whether traffic is malicious or benign.

In [ ]:
# --- Step 4.5: Remove Duplicates to match Paper ---
# The paper removed ~307k duplicates[cite: 318].
# We drop duplicates based on ALL columns except the Flow ID (which is unique for every row)

print(f"Shape BEFORE dropping duplicates: {df_full.shape}")

# 1. Drop Flow ID if it exists (it makes rows look unique when they aren't)
if 'Flow ID' in df_full.columns:
    df_full.drop(columns=['Flow ID'], inplace=True)

# 2. Drop duplicates
# We keep the first occurrence and drop the rest
df_full.drop_duplicates(inplace=True)

print(f"Shape AFTER dropping duplicates: {df_full.shape}")
print("\nNew Label Counts (Should match Paper Table 3):")
print(df_full['Label'].value_counts())

Shape BEFORE dropping duplicates: (2827876, 85)
Shape AFTER dropping duplicates: (2827677, 84)

New Label Counts (Should match Paper Table 3):
Label
BENIGN                        2271122
DoS Hulk                       230123
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack  Brute Force         1507
Web Attack  XSS                  652
Infiltration                       36
Web Attack  Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


# 4️⃣ Phase 4: Data Sanitization & Integrity Checks
**Objective:** Ensure the dataset contains only valid, logical numerical data.

### A. Handling Missing and Infinite Values
Network capture tools may produce `Infinity` or `NaN` (Not a Number) errors during division-by-zero events. We identify and remove these invalid rows.

### B. Handling Impossible Values
The statistical description revealed **negative values** in time-based columns like `Flow Duration` and `Flow IAT Mean`. A time duration cannot be negative; this implies a logging error where a flow essentially "ended before it started." These are unreliable records that must be purged to maintain data quality.

### Checking NULLS

In [ ]:
rows_all_nan = df_full[df_full.isna().all(axis=1)]
print("Rows with all null:", rows_all_nan.shape[0])

Rows with all null: 288602


## Dropping all nulls rows in the dataframe

In [ ]:
df_full = df_full.dropna(how='all')

In [ ]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2830743 entries, 0 to 3119344
Data columns (total 85 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   Flow ID                      object 
 1   Source IP                    object 
 2   Source Port                  float64
 3   Destination IP               object 
 4   Destination Port             float64
 5   Protocol                     float64
 6   Timestamp                    object 
 7   Flow Duration                float64
 8   Total Fwd Packets            float64
 9   Total Backward Packets       float64
 10  Total Length of Fwd Packets  float64
 11  Total Length of Bwd Packets  float64
 12  Fwd Packet Length Max        float64
 13  Fwd Packet Length Min        float64
 14  Fwd Packet Length Mean       float64
 15  Fwd Packet Length Std        float64
 16  Bwd Packet Length Max        float64
 17  Bwd Packet Length Min        float64
 18  Bwd Packet Length Mean       float64
 19  Bwd P

In [ ]:
df_full.isnull().sum()/len(df_full)*100

,0
Flow ID,0.000000
Source IP,0.000000
Source Port,0.000000
Destination IP,0.000000
Destination Port,0.000000
Protocol,0.000000
Timestamp,0.000000
Flow Duration,0.000000
Total Fwd Packets,0.000000
Total Backward Packets,0.000000


***0.047973 % are Nulls in Flow Bytes/s coloumn so we drop nulls for an easier approach since its a very small percentage***

In [ ]:
df_full.dropna(inplace=True)

In [ ]:
df_full.isna().sum().sum()

np.int64(0)

## Checking and handling Infinite Values



In [ ]:
# Replace infinite values with NaN for easy dropping
df_full.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check for any remaining NaN values after replacing inf
print("Number of NaN values after replacing infinite:", df_full.isnull().sum().sum())

# Drop rows with NaN values (which now include the original infinite values)
df_full.dropna(inplace=True)

print("Shape after removing infinite values:", df_full.shape)

Number of NaN values after replacing infinite: 3018
Shape after removing infinite values: (2827876, 85)


## Identifying and Handling Impossible Values

The `.describe()` output revealed impossible negative values in time-based columns like `Flow Duration` (min: -13) and `Flow IAT Mean` (min: -13). A time duration cannot be negative, as this would imply a flow ended before it started.

[cite_start]These are clearly "unreliable values" [cite: 174, 184] and data errors, likely from the capture or calculation process. We must remove these rows to ensure data quality. We will **not** remove rows with a value of 0, as a zero duration is a valid measurement for a single-packet flow.

In [ ]:
df_full.describe()

,Source Port,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
count,2.520265e+06,2.520265e+06,2.520265e+06,2520265,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2520265.0,2.520265e+06,2520265.0,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2520265.0,2520265.0,2520265.0,2520265.0,2520265.0,2520265.0,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06,2.520265e+06
mean,4.045458e+04,8.690402e+03,9.966154e+00,2017-05-09 09:45:56.949876736,1.659512e+07,1.028425e+01,1.157504e+01,6.121440e+02,1.814824e+04,2.312822e+02,1.921161e+01,6.352250e+01,7.733986e+01,9.751774e+02,4.316724e+01,3.406932e+02,3.766249e+02,1.413723e+06,4.731932e+04,1.446448e+06,3.278846e+06,1.030166e+07,1.704576e+05,1.625527e+07,2.919875e+06,3.669416e+06,1.014508e+07,1.135911e+06,1.111260e+07,2.028163e+06,1.669034e+06,5.261732e+06,1.086341e+06,4.867861e-02,0.0,3.174269e-05,0.0,-2.920640e+04,-2.555781e+03,4.089038e+04,6.510272e+03,1.683962e+01,1.063978e+03,1.906989e+02,3.302700e+02,5.458915e+05,3.210218e-02,4.867861e-02,2.721936e-04,2.974477e-01,3.119049e-01,1.014203e-01,3.174269e-05,2.733839e-04,7.003827e-01,2.124845e+02,6.352250e+01,3.406932e+02,-2.920640e+04,0.0,0.0,0.0,0.0,0.0,0.0,1.028425e+01,6.121323e+02,1.157504e+01,1.814786e+04,7.266990e+03,2.229197e+03,6.011345e+00,-3.082890e+03,9.159784e+04,4.620154e+04,1.720535e+05,6.547743e+04,9.339342e+06,5.659138e+05,9.765835e+06,8.894551e+06
min,0.000000e+00,0.000000e+00,0.000000e+00,2017-03-07 01:00:01,-1.300000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-2.610000e+08,-2.000000e+06,-1.300000e+01,0.000000e+00,-1.300000e+01,-1.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,-3.221223e+10,-1.073741e+09,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0

In [ ]:
print(f"Original shape before removing negative durations: {df_full.shape}")

negative_rows = df_full[
    (df_full['Flow Duration'] < 0) |
    (df_full['Flow IAT Mean'] < 0)
].shape[0]

print(f"Found {negative_rows} rows with negative time values.")

df_full = df_full[df_full['Flow Duration'] >= 0]
df_full = df_full[df_full['Flow IAT Mean'] >= 0]

print(f"New shape after removing negative time values: {df_full.shape}")

Original shape before removing negative durations: (2520265, 84)
Found 107 rows with negative time values.
New shape after removing negative time values: (2520158, 84)


# 5️⃣ Phase 6: Duplicate Removal (Exact Matches)
**Objective:** Eliminate redundant data recording errors without altering the nature of high-volume attacks.  
**Action:** We perform a strict deduplication, removing rows **only** if they are identical in every single feature (including timestamps and ports). This approach preserves legitimate high-frequency traffic, such as PortScan attacks or DDoS floods, where identical packets may be sent at different times or to different ports.

## Removing Duplicates

In [ ]:
# --- 1. Define Metadata (Columns to IGNORE when checking for duplicates) ---

# OLD LIST (Strict behavioral cleaning)
# metadata_cols = ['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Timestamp']

# NEW LIST (Matches the Paper's high numbers)
# We remove 'Timestamp' from this list.
# Now, pandas will see (Packet A at 10:00) and (Packet A at 10:01) as DIFFERENT rows.
metadata_cols = ['Source IP', 'Source Port', 'Destination IP','Timestamp', 'Protocol']
all_cols = df_full.columns.tolist()
feature_cols = [col for col in all_cols if col not in metadata_cols and col != 'Label']

print(f"Checking for duplicates based on {len(feature_cols)} behavioral features.")

# 3. Check the "before" count
rows_before = df_full.shape[0]
print(f"Rows before dropping behavioral duplicates: {rows_before}")

# 4. Drop duplicates, looking ONLY at the 'subset' of feature_cols
df_full.drop_duplicates(subset=feature_cols, inplace=True)

# 5. Check the "after" count
rows_after = df_full.shape[0]
print(f"Rows after dropping behavioral duplicates: {rows_after}")
print(f"Total behavioral duplicates removed: {rows_before - rows_after}")

Checking for duplicates based on 78 behavioral features.
Rows before dropping behavioral duplicates: 2827677
Rows after dropping behavioral duplicates: 2520265
Total behavioral duplicates removed: 307412


In [ ]:
df_full.drop_duplicates(inplace=True)

# 6️⃣ Phase 7: Timestamp Standardization
**Objective:** Convert temporal data into a usable format for analysis  
**Action:** The raw `Timestamp` column is currently stored as a string object. We convert it to a standard Python `datetime` object to enable chronological sorting and time-based feature extraction in later stages.

## Convert Timestamp to Datetime Objects

The `Timestamp` column is currently an object type. Converting it to datetime objects will enable time-series analysis and extraction of time-based features.

In [ ]:
print("Converting Timestamp column to datetime...")
try:

    df_full['Timestamp'] = pd.to_datetime(df_full['Timestamp'],
                                          format='mixed',
                                          dayfirst=False,
                                          errors='coerce')

    print("Timestamp conversion successful.")

    # Check the Dtype
    df_full.info(verbose=True)

except Exception as e:
    print(f"Error converting Timestamp: {e}")

Converting Timestamp column to datetime...
Timestamp conversion successful.
<class 'pandas.core.frame.DataFrame'>
Index: 2520265 entries, 0 to 3119344
Data columns (total 84 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   Source IP                    object        
 1   Source Port                  float64       
 2   Destination IP               object        
 3   Destination Port             float64       
 4   Protocol                     float64       
 5   Timestamp                    datetime64[ns]
 6   Flow Duration                float64       
 7   Total Fwd Packets            float64       
 8   Total Backward Packets       float64       
 9   Total Length of Fwd Packets  float64       
 10  Total Length of Bwd Packets  float64       
 11  Fwd Packet Length Max        float64       
 12  Fwd Packet Length Min        float64       
 13  Fwd Packet Length Mean       float64       
 14  Fwd Packet Length Std      

In [ ]:
attack_hours = df_full['Timestamp'].dt.hour.value_counts().sort_index()

# 7️⃣ Phase 8: Memory Optimization
**Objective:** Reduce the dataset's memory footprint to improve processing speed.  
**Action:** The raw data uses 64-bit integers by default. We scan the numerical columns and "downcast" integers to the smallest possible type (e.g., `int8` or `int16`) that can hold the values.
*Note: We maintain `float` columns as 64-bit to ensure precision is not lost in statistical features.*

## Optimize Data Types

Based on the paper's methodology, we will optimize the data types to reduce memory usage. This involves downcasting integer columns to smaller integer types while keeping float columns as `float64`.

In [ ]:
# 6. Optimize Data Types (Revised Method)
print("\nOptimizing data types (Revised Method)...")

# Check memory usage before
mem_before = df_full.memory_usage(deep=True).sum() / (1024**2)
print(f"Memory usage before downcasting: {mem_before:.2f} MB")

# Get all float columns
float_cols = df_full.select_dtypes(include=['float64']).columns
downcasted_cols = []

for col in float_cols:
    try:
        # Check if all values in the column are whole numbers
        # We do this by checking if the fractional part of all numbers is 0
        if (np.modf(df_full[col])[0] == 0).all():
            # If it's all whole numbers, downcast it to integer
            df_full[col] = pd.to_numeric(df_full[col], downcast='integer')
            downcasted_cols.append(col)
    except TypeError:
        # This can happen if there's an unexpected value; we'll just skip it
        print(f"Could not check column {col}, skipping.")

print(f"Successfully downcasted {len(downcasted_cols)} integer-like columns.")
print("True float columns (with decimals) were maintained.")

# Check memory usage after
mem_after = df_full.memory_usage(deep=True).sum() / (1024**2)
print(f"Memory usage after downcasting: {mem_after:.2f} MB")


Optimizing data types (Revised Method)...
Memory usage before downcasting: 2004.22 MB


/usr/local/lib/python3.12/dist-packages/pandas/core/dtypes/cast.py:378: RuntimeWarning: invalid value encountered in cast
  new_result = trans(result).astype(dtype)


Successfully downcasted 56 integer-like columns.
True float columns (with decimals) were maintained.
Memory usage after downcasting: 1314.44 MB


In [ ]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2520158 entries, 0 to 3119344
Data columns (total 84 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   Source IP                    object        
 1   Source Port                  int32         
 2   Destination IP               object        
 3   Destination Port             int32         
 4   Protocol                     int8          
 5   Timestamp                    datetime64[ns]
 6   Flow Duration                int32         
 7   Total Fwd Packets            int32         
 8   Total Backward Packets       int32         
 9   Total Length of Fwd Packets  int32         
 10  Total Length of Bwd Packets  int32         
 11  Fwd Packet Length Max        int16         
 12  Fwd Packet Length Min        int16         
 13  Fwd Packet Length Mean       float64       
 14  Fwd Packet Length Std        float64       
 15  Bwd Packet Length Max        int16         
 16  Bwd P

# 8️⃣ Phase 9: Label Verification
**Objective:** Confirm the consistency of the target variable.
**Action:** We inspect the `Label` column to ensure all traffic types are correctly named and to get a preliminary understanding of the class distribution (Benign vs. Attacks).

In [ ]:
df_full['Label'].value_counts()

,count
Label,
BENIGN,2272688
DoS Hulk,230124
PortScan,158930
DDoS,128027
DoS GoldenEye,10293
FTP-Patator,7938
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499


# 9️⃣ Phase 10: Save Final Dataset
**Objective:** Export the fully cleaned and optimized dataset.
**Action:** We save the DataFrame to **Parquet** format. Parquet is chosen over CSV because it preserves the specific data types (like `int8` and `datetime`) we just optimized and offers significantly faster read/write speeds.

## Saving Final Cleaned File

In [ ]:

parquet_path = "/content/drive/MyDrive/Depi Project/Cleaned Data/Cleaned_Data.parquet"
df_full.to_parquet(parquet_path, index=False)

print(f"Cleaned data saved correctly to {parquet_path}")

Cleaned data saved correctly to /content/drive/MyDrive/Depi Project/Cleaned Data/Cleaned_Data.parquet
